In [0]:
import os
import pandas as pd
import z_pipelineUtils as z_pU
import z_schemas as z_s
import z_tests as z_t

# File that stores the most recent run details of the pipeline
PIPELINE_STATE = "/Volumes/nc_pipeline/default/pipeline_data/pipeline_state.json"
# Directory that will store the unzipped files
UNZIP_DEST = "/Volumes/nc_pipeline/default/pipeline_data/unzipped"
# Encoding on the voter files
FILE_ENCODING = "latin-1"

saved_state, live_state = z_pU.readStateJson(PIPELINE_STATE)
pre_tests = z_t.PRE_TESTS

for voter_file in saved_state:
    downloaded = saved_state[voter_file]["downloaded"]
    unzipped = saved_state[voter_file]["unzipped"]
    tested = saved_state[voter_file]["tested"]
    data_filename = voter_file.replace(".zip", ".txt")
    full_file_path = os.path.join(UNZIP_DEST, data_filename)
    print(f"Loading {voter_file}...")
    v_df = pd.read_csv(full_file_path, delimiter="\t", dtype="str", quotechar='"', encoding=FILE_ENCODING)
    table_name = z_s.SCHEMAS[voter_file]["table_name"]
    if (downloaded is True) and (unzipped is True) and (tested is False):
        print(f"Testing {voter_file}...")
        test_results = {}
        for test in pre_tests[table_name]:
            func = getattr(z_pU, test["function"])
            result = func(v_df, voter_file, **test["params"])
            test_results[test["function"]] = result
        if False not in test_results.values():
            print(f"All tests passed.")
            live_state[voter_file]["tested"] = True
            z_pU.updateStateJson(live_state, PIPELINE_STATE)
        else:
            print("Test(s) failed.")
            for func, result in test_results.items():
                print(f"{func}: {result}")
    elif tested is True:
        print(f"Newest dataset for {voter_file} has already been tested.")
    else:
        print(f"{voter_file} is not ready for testing (downloaded={downloaded}, unzipped={unzipped}).")
    print()
